# SSN Optimizer Smoke Tests

This notebook checks the current `SignedModel` interface for the unified semismooth-Newton optimizer. `optimizer="SSN"` uses the line-search/Levenberg-Marquardt strategy and `optimizer="SSN_TR"` selects the trust-region/Steihaug-CG strategy inside the same `SSN` optimizer implementation.

In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'pyproject.toml').exists():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(REPO_ROOT)

/Users/chaoruiz/Documents/Repos/SparseNNforHJB


In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import torch

from src.logging_config import configure_logging
from src.models import SignedModel

configure_logging(verbose=False)
torch.set_default_dtype(torch.float64)

## Helpers

In [3]:
ALPHA = 1e-5
GAMMA = 5.0
POWER = 2.1
TRAINING_FRACTION = 0.8
ITERATIONS = 2
DISPLAY_EVERY = 1

def load_structured_dataset(path, *, limit=None, seed=0):
    raw = np.load(path)
    if limit is not None and limit < len(raw):
        rng = np.random.default_rng(seed)
        raw = raw[rng.permutation(len(raw))[:limit]]
    return {
        'x': np.asarray(raw['x'], dtype=np.float64),
        'v': np.asarray(raw['v'], dtype=np.float64),
        'dv': np.asarray(raw['dv'], dtype=np.float64),
    }

def fit_outer_weights(data, optimizer):
    model = SignedModel(
        alpha=ALPHA,
        gamma=GAMMA,
        optimizer=optimizer,
        activation=torch.relu,
        power=POWER,
        training_percentage=TRAINING_FRACTION,
        train_outerweights=True,
        verbose=False,
    )
    data_train, data_valid = model._prepare_data(data)
    made_progress = model.train(
        data_train,
        data_valid,
        iterations=ITERATIONS,
        display_every=DISPLAY_EVERY,
    )
    return model, made_progress

def summarize_run(label, model, made_progress):
    return {
        'run': label,
        'made_progress': bool(made_progress),
        'train_loss_last': model.loss_history['train_loss'][-1],
        'val_loss_last': model.loss_history['val_loss'][-1],
        'history_points': len(model.loss_history['train_loss']),
    }

## 1D Case: Gaussian Window

In [4]:
gaussian_data = load_structured_dataset(
    REPO_ROOT / 'rawdata/data/gaussian_window_1000.npy',
    limit=80,
)
gaussian_data['x'].shape, gaussian_data['v'].shape, gaussian_data['dv'].shape

((80, 1), (80,), (80, 1))

In [5]:
gaussian_results = []
for optimizer in ('SSN', 'SSN_TR'):
    model, made_progress = fit_outer_weights(gaussian_data, optimizer)
    gaussian_results.append(summarize_run(f'gaussian/{optimizer}', model, made_progress))
gaussian_results

[{'run': 'gaussian/SSN',
  'made_progress': True,
  'train_loss_last': 1.0140819016602731,
  'val_loss_last': 1.9752359857292008,
  'history_points': 2},
 {'run': 'gaussian/SSN_TR',
  'made_progress': True,
  'train_loss_last': 1.7226866087831287,
  'val_loss_last': 2.266838183661751,
  'history_points': 2}]

## 2D Case: Van der Pol Value Data

In [6]:
vdp_data = load_structured_dataset(
    REPO_ROOT / 'rawdata/data/VDP_beta_0.1_grid_30x30.npy',
    limit=120,
)
vdp_data['x'].shape, vdp_data['v'].shape, vdp_data['dv'].shape

((120, 2), (120,), (120, 2))

In [7]:
vdp_results = []
for optimizer in ('SSN', 'SSN_TR'):
    model, made_progress = fit_outer_weights(vdp_data, optimizer)
    vdp_results.append(summarize_run(f'vdp/{optimizer}', model, made_progress))
vdp_results

[{'run': 'vdp/SSN',
  'made_progress': True,
  'train_loss_last': 0.9253993673325671,
  'val_loss_last': 0.9270758864486643,
  'history_points': 2},
 {'run': 'vdp/SSN_TR',
  'made_progress': True,
  'train_loss_last': 12.78571876172414,
  'val_loss_last': 9.76430956048693,
  'history_points': 2}]

## Notes

This notebook is intentionally a fast smoke test. It verifies that the current model API can prepare data and run both SSN globalization strategies on small deterministic subsets. Longer optimizer-quality comparisons should live in scripts or experiment run records rather than in this notebook.

## PDAP model 

In [8]:
from datetime import date
run_date = date.today().strftime("%Y%m%d")
configure_logging(verbose=True)

<Logger nnforhjb (INFO)>

In [9]:
path = 'rawdata/data/VDP_beta_0.1_grid_30x30.npy'
data = np.load(path)
len(data)

data_dict = {
    "x": np.asarray(data["x"], dtype=np.float64),    # shape (N, 2)
    "v": np.asarray(data["v"], dtype=np.float64),    # shape (N,)
    "dv": np.asarray(data["dv"], dtype=np.float64),  # shape (N, 2)
}

In [10]:
import random

seed = 42
gammas = [0] 
alpha = 1e-5
power = 1.0
activation = torch.tanh
use_sphere = False

num_iterations = 2
num_insertions = 50
pruning_threshold = 1e-5

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)  # if you ever use GPU
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [11]:
from src.PDAP import PDAP
results_l2 = []

for gamma in gammas:
    pdpa = PDAP(
        data=data_dict,
        alpha=alpha,
        gamma=gamma,
        power=power,
        model="signed",
        insertion="profile",
        activation=activation,
        use_sphere=use_sphere,
        loss_weights="l2",
        verbose=True,
    )

    result = pdpa.fit(
        num_iterations=num_iterations,
        num_insertion=num_insertions,
        threshold=pruning_threshold,
        verbose=True,
    )
    results_l2.append(result)

INFO PDAP run
INFO   +------------------+--------------------------+
INFO   | model            | SignedModel              |
INFO   | insertion rule   | profile                  |
INFO   | samples          | 899 train, 1 validation  |
INFO   | input dimension  | 2                        |
INFO   | alpha            | 1.00e-05                 |
INFO   | gamma            | 0.00e+00                 |
INFO   | activation power | 1                        |
INFO   +------------------+--------------------------+
INFO Progress
INFO   +---------+---------+--------------+--------------+------------+------------+
INFO   | iter    | neurons |   train loss |     val loss |     val L2 |     val H1 |
INFO   +---------+---------+--------------+--------------+------------+------------+
INFO   | 1/2     |       2 |    1.414e+00 |    1.505e+01 |  7.081e-01 |  8.936e-01 |
INFO   | 2/2     |       8 |    7.507e-01 |    9.982e+00 |  5.767e-01 |  8.548e-01 |
INFO   +---------+---------+--------------+---------